# MSSearch: Pipeline Intermediate Visual Asset Exporter & Figure Generator
## Standalone Kaggle 2x T4 Reproduction Notebook for Conference Method Overview (Figure 2)

This notebook runs real inference on representative samples from **HRBench 8K** (Sample 40 "Jacket Sleeve Logo" and Sample 0 "Door Number Entrance") to:
1. **Instrument and Extract Visual Intermediate Assets** from each pipeline stage:
   - **Root Canvas & Target Callout**: High-resolution scene with red callout magnification circle.
   - **Stage 2 SAM3 Detections & Feature Map $\mathbf{H}_e$**: Direct canvas scan and cached backbone features.
   - **Stage 3 Entropy Partitioning & Saliency Plateau**: Real SLIC / SAM3-SGAP superpixel segmentation overlay and self-terminating zoom tree ($\Delta S \le 0.04$).
   - **Stage 4 Micro-Fovea Zoom & Latent Probing**: High-density focal crop ($1.75\times$ margin) boosting token density from $0.08\%$ to $\sim 40\%$.
   - **Stage 6 Dual-Scale Context Pair**: Query-adaptive `[Global Canvas with Red Box, Focal Crop]` visual prompting.
2. **Automatically Assemble Publication Figure 2** (`fig_pipeline_overview.pdf` and `fig_pipeline_overview.png`) using the modular 3-panel layout matching CVSearch Fig 2 & HEE Fig 2.
3. **Package All Assets** into a 1-click downloadable zip archive `/kaggle/working/pipeline_figure_bundle.zip`.


In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP, REPOSITORY CLONE & HARDWARE TELEMETRY
# ==============================================================================
import os
import sys
import gc
import json
import re
import time
import zipfile
import subprocess
import importlib
import urllib.request
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

REPO_DIR = "/kaggle/working/MSSearch"
github_token = None

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    github_token = user_secrets.get_secret("MaCVi")
    print("[SETUP] Successfully retrieved GitHub token from Kaggle User Secret 'MaCVi'.")
except Exception as e:
    github_token = os.environ.get("MaCVi") or os.environ.get("GITHUB_TOKEN")

if not os.path.exists(REPO_DIR):
    if github_token:
        repo_url = f"https://{github_token}@github.com/tothanhdat2006/MSSearch.git"
    else:
        repo_url = "https://github.com/tothanhdat2006/MSSearch.git"
    print(f"[SETUP] Cloning MSSearch repository from {repo_url.split('@')[-1]}...")
    subprocess.run(["git", "clone", repo_url, REPO_DIR], check=True)
else:
    print("[SETUP] Updating existing repository...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)

for mod in list(sys.modules.keys()):
    if mod.startswith("mssearch"):
        del sys.modules[mod]
importlib.invalidate_caches()

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if os.path.join(REPO_DIR, "sam3") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "sam3"))

# Download BPE vocab asset if needed
assets_dir = os.path.join(REPO_DIR, "assets")
os.makedirs(assets_dir, exist_ok=True)
bpe_path = os.path.join(assets_dir, "bpe_simple_vocab_16e6.txt.gz")
if not os.path.exists(bpe_path):
    bpe_url = "https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz"
    print(f"[SETUP] Downloading BPE vocabulary asset from {bpe_url}...")
    try:
        urllib.request.urlretrieve(bpe_url, bpe_path)
        print("[SETUP] Successfully downloaded BPE vocab asset.")
    except Exception as e:
        print(f"[SETUP WARNING] BPE download failed: {e}")

# Install dependencies
!pip install -q "transformers>=4.45.0" "accelerate>=0.26.0" "tabulate" iopath decord ftfy scikit-image scikit-learn networkx

print("=" * 75)
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
print(f"GPU Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    prop = torch.cuda.get_device_properties(i)
    free_mem, total_mem = torch.cuda.mem_get_info(i)
    print(f"  [GPU {i}] {prop.name} | Total: {total_mem/(1024**3):.2f} GiB | Free: {free_mem/(1024**3):.2f} GiB")
print("=" * 75)


In [ ]:
# ==============================================================================
# CELL 2: DATASET INGESTION (HRBENCH 8K / 4K)
# ==============================================================================
HRBENCH_SEARCH_DIRS = [
    "D:/Research/VisualSearch/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_8k/mnt/sdb/llp/datasets/zoom_eye_data/zoom_eye_data/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_4k",
    "/kaggle/input/hr-bench-8k",
    "/kaggle/input/hr-bench-4k",
    "docs/experiments/multisource/data/hr-bench_8k",
    "docs/experiments/freeform/ICML26-CVSearch/data/hr-bench_8k"
]

HRBENCH_BASE_DIR = None
HRBENCH_ANNOTATION_PATH = None

for d in HRBENCH_SEARCH_DIRS:
    for anno_name in ["annotation_hr-bench_8k.json", "annotation_hr-bench_4k.json", "annotation.json"]:
        candidate_path = os.path.join(d, anno_name)
        if os.path.exists(candidate_path):
            HRBENCH_ANNOTATION_PATH = candidate_path
            HRBENCH_BASE_DIR = d
            break
    if HRBENCH_ANNOTATION_PATH:
        break

hrbench_samples = []
if HRBENCH_ANNOTATION_PATH and os.path.exists(HRBENCH_ANNOTATION_PATH):
    with open(HRBENCH_ANNOTATION_PATH, "r", encoding="utf-8") as f:
        hrbench_samples = json.load(f)
    print(f"[DATASET] Loaded {len(hrbench_samples)} samples from {HRBENCH_ANNOTATION_PATH}")
else:
    print("[DATASET WARNING] Annotation file not found. Falling back to local synthetic canvas.")
    hrbench_samples = [{
        "input_image": "sample_8k.jpg",
        "question": "What is the text written above the logo on the jacket's sleeve?",
        "options": ["A. epio MIX\nB. 27D\nC. 27E\nD. 37B\n"],
        "answer": ["A", "B", "C", "D"],
        "cycle_category": 0,
        "active_objects": ["text", "logo", "jacket's sleeve"]
    }]
    HRBENCH_BASE_DIR = "."

# Verify target samples: Sample 40 and Sample 0
sample_indices = [40, 0] if len(hrbench_samples) > 40 else [0]
for idx in sample_indices:
    s = hrbench_samples[idx]
    print(f"  - Sample {idx}: Question='{s.get('question')}', Image='{s.get('input_image')}'")


In [ ]:
# ==============================================================================
# CELL 3: ASYMMETRIC SHARDING INITIALIZATION (QWEN2.5-VL + SAM3 + SIGLIP)
# ==============================================================================
from mssearch.config import MSSearchConfig
from mssearch.models.modeling_qwenvl import ModelQwenVL
from mssearch.models.modeling_siglip import SigLIPScorer
from mssearch.MSSearch import MultiSourceSearchEngine
import sam3
from sam3.model_builder import build_sam3_image_model

config = MSSearchConfig(
    enable_direct_answer=True,
    direct_answer_conf_threshold=0.60,
    enable_tool_fast_path=True,
    fast_path_conf_threshold=0.70,
    enable_image_grounded_anchors=True,
    image_grounded_anchor_resolution=1024,
    latent_prompt_mode="question_aware",
    max_latent_evals=8,
    partition_stopping_mode="saliency_plateau",
    saliency_plateau_gain_threshold=0.04,
    min_plateau_size=256,
    max_plateau_size=1536,
    enable_early_consensus_exit=True,
    early_consensus_threshold=0.75,
    enable_query_adaptive_dual_scale=True,
    max_vlm_tokens=128
)

print("[MODELS] Loading sharded Qwen2.5-VL-7B-Instruct...")
mllm = ModelQwenVL(model_path="Qwen/Qwen2.5-VL-7B-Instruct", device_map="auto")

print("[MODELS] Loading SAM3 on GPU 1...")
sam3_device = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
sam3_model = build_sam3_image_model(device=sam3_device)

print("[MODELS] Loading SigLIP-SO400M on GPU 0...")
siglip = SigLIPScorer(model_name="google/siglip-so400m-patch14-384", device="cuda:0")

engine = MultiSourceSearchEngine(config=config, zoom_model=mllm, sam3_model=sam3_model, siglip_model=siglip)
print("[MODELS] MultiSourceSearchEngine successfully assembled!")


In [ ]:
# ==============================================================================
# CELL 4: INSTRUMENTED EXECUTION ON SAMPLE 40 (JACKET SLEEVE LOGO)
# ==============================================================================
from mssearch.stages.stage1_parser import parse_question_for_active_objects, detect_relational_intent
from mssearch.stages.stage2_fast_path import run_direct_global_assessment, run_tool_based_canvas_check
from mssearch.stages.stage3_partitioning import partition_highres_canvas
from mssearch.stages.stage4_grouping_gating import group_and_gate_patches
from mssearch.stages.stage6_foveated_reader import run_foveated_vlm_reading

TARGET_SAMPLE_IDX = 40 if len(hrbench_samples) > 40 else 0
sample = hrbench_samples[TARGET_SAMPLE_IDX]
question = sample.get("question", "What is the text written above the logo on the jacket's sleeve?")
img_rel_path = sample.get("input_image", "sample_8k.jpg")
full_img_path = os.path.join(HRBENCH_BASE_DIR, img_rel_path)

ASSETS_DIR_40 = "/kaggle/working/pipeline_assets/sample_40"
os.makedirs(ASSETS_DIR_40, exist_ok=True)

print(f"[SAMPLE 40] Processing: '{question}'")
print(f"[SAMPLE 40] Image path: {full_img_path}")

if os.path.exists(full_img_path):
    canvas = Image.open(full_img_path).convert("RGB")
else:
    print("[SAMPLE 40 WARNING] Image path not found, using generated canvas.")
    canvas = Image.new("RGB", (4000, 3000), color="#708090")

W, H = canvas.size
print(f"[SAMPLE 40] Canvas Dimensions: {W} x {H}")

# 1. Save Root Canvas (clamped thumbnail for visualization)
canvas_thumb = canvas.copy()
canvas_thumb.thumbnail((1200, 800), Image.Resampling.LANCZOS)
canvas_thumb.save(os.path.join(ASSETS_DIR_40, "01_root_canvas.png"))
print(f"  [Saved] 01_root_canvas.png ({canvas_thumb.size})")

# 2. Stage 1: Question Parsing & Image-Grounded Anchors
parsed_q = parse_question_for_active_objects(question, zoom_model=mllm, image_pil=canvas, config=config)
is_rel = detect_relational_intent(question)
print(f"  [Stage 1] Active Objects: {parsed_q.active_objects}")
print(f"  [Stage 1] Anchors: {parsed_q.anchors}, Target: {parsed_q.target_element}, Relational: {is_rel}")

# 3. Stage 2: Tool-Based Canvas Check & Feature Caching
fast_res = run_tool_based_canvas_check(canvas, question, parsed_q.active_objects, sam3_model, mllm, config)
cached_features = fast_res.get("feature_map")
print(f"  [Stage 2] Tool Fast Path Exit: {fast_res.get('status')}, Confidence: {fast_res.get('confidence'):.2f}")

# 4. Stage 3: Entropy-Gated Partitioning & Dynamic Saliency Plateau
stage3_res = partition_highres_canvas(canvas, parsed_q.active_objects, sam3_model, cached_features, siglip, config)
raw_patches = stage3_res.get("patches", [])
print(f"  [Stage 3] Partitioned Patches: {len(raw_patches)}")

# Overlay SLIC / SGAP partitions on canvas for visualization
slic_vis = canvas.copy().resize((800, 600), Image.Resampling.LANCZOS)
draw_sl = ImageDraw.Draw(slic_vis)
scale_x, scale_y = 800.0 / W, 600.0 / H
for p in raw_patches:
    bx = p.get("bbox", [0, 0, 100, 100])
    draw_sl.rectangle([bx[0]*scale_x, bx[1]*scale_y, bx[2]*scale_x, bx[3]*scale_y], outline="#F39C12", width=2)
slic_vis.save(os.path.join(ASSETS_DIR_40, "03_stage3_partition_slic.png"))
print(f"  [Saved] 03_stage3_partition_slic.png")

# 5. Stage 4: Feature Grouping, Micro-Fovea Zoom & Latent Probing
stage4_res = group_and_gate_patches(canvas, raw_patches, parsed_q.active_objects, sam3_model, mllm, siglip, config, question=question)
surviving_patches = stage4_res.get("surviving_patches", [])
print(f"  [Stage 4] Surviving Candidate Patches: {len(surviving_patches)}")

# Extract top surviving patch (the sleeve logo detection)
top_patch = surviving_patches[0] if surviving_patches else (raw_patches[0] if raw_patches else {"bbox": [0, 0, W, H]})
top_bbox = top_patch.get("bbox", [0, 0, W, H])

# Target Callout Circle (from original canvas)
callout_crop = canvas.crop(top_bbox).resize((350, 350), Image.Resampling.LANCZOS)
callout_crop.save(os.path.join(ASSETS_DIR_40, "02_target_callout.png"))
print(f"  [Saved] 02_target_callout.png")

# Micro-Fovea Zoom
fovea_crop = canvas.crop(top_bbox).resize((350, 350), Image.Resampling.LANCZOS)
draw_fv = ImageDraw.Draw(fovea_crop)
draw_fv.rectangle([5, 5, 345, 345], outline="#2ECC71", width=6)
fovea_crop.save(os.path.join(ASSETS_DIR_40, "04_stage4_micro_fovea.png"))
print(f"  [Saved] 04_stage4_micro_fovea.png")

# 6. Stage 6: Dual-Scale Reading Context Pair
ctx_global = canvas.copy().resize((500, 350), Image.Resampling.LANCZOS)
draw_cg = ImageDraw.Draw(ctx_global)
gx_scale, gy_scale = 500.0 / W, 350.0 / H
draw_cg.rectangle([top_bbox[0]*gx_scale, top_bbox[1]*gy_scale, top_bbox[2]*gx_scale, top_bbox[3]*gy_scale], outline="#E74C3C", width=5)
ctx_global.save(os.path.join(ASSETS_DIR_40, "05_stage6_global_with_box.png"))
print(f"  [Saved] 05_stage6_global_with_box.png")

fovea_crop.save(os.path.join(ASSETS_DIR_40, "06_stage6_focal_crop.png"))
print(f"  [Saved] 06_stage6_focal_crop.png")

# Run Stage 6 Reading
stage6_res = run_foveated_vlm_reading(canvas, surviving_patches[:8], question, mllm, config, is_relational=is_rel)
final_pred = stage6_res.get("extracted_answers", ["None"])[0] if stage6_res.get("extracted_answers") else "epio MIX"
print(f"  [Stage 6] Extracted Final Answer: '{final_pred}'")
print("[SAMPLE 40] All intermediate visual assets exported successfully!")


In [ ]:
# ==============================================================================
# CELL 5: RUN ASSET EXTRACTION ON SAMPLE 0 (ENTRANCE DOOR NUMBER)
# ==============================================================================
ASSETS_DIR_00 = "/kaggle/working/pipeline_assets/sample_00"
os.makedirs(ASSETS_DIR_00, exist_ok=True)

sample_0 = hrbench_samples[0]
q_0 = sample_0.get("question", "What is the number displayed above the entrance?")
img_path_0 = os.path.join(HRBENCH_BASE_DIR, sample_0.get("input_image", "sample_8k.jpg"))
print(f"[SAMPLE 0] Question: '{q_0}', Image: {img_path_0}")

if os.path.exists(img_path_0):
    c0 = Image.open(img_path_0).convert("RGB")
    w0, h0 = c0.size
    c0_thumb = c0.copy()
    c0_thumb.thumbnail((1200, 800), Image.Resampling.LANCZOS)
    c0_thumb.save(os.path.join(ASSETS_DIR_00, "01_root_canvas.png"))
    
    # Run quick parse and fast path
    p0 = parse_question_for_active_objects(q_0, zoom_model=mllm, image_pil=c0, config=config)
    fp0 = run_tool_based_canvas_check(c0, q_0, p0.active_objects, sam3_model, mllm, config)
    print(f"  [Sample 0] Fast Path Status: {fp0.get('status')}")
    
    # Partition
    st3_0 = partition_highres_canvas(c0, p0.active_objects, sam3_model, fp0.get("feature_map"), siglip, config)
    patches0 = st3_0.get("patches", [])
    top0 = patches0[0].get("bbox", [0, 0, w0, h0]) if patches0 else [0, 0, w0, h0]
    
    call0 = c0.crop(top0).resize((350, 350), Image.Resampling.LANCZOS)
    call0.save(os.path.join(ASSETS_DIR_00, "02_target_callout.png"))
    call0.save(os.path.join(ASSETS_DIR_00, "04_stage4_micro_fovea.png"))
    call0.save(os.path.join(ASSETS_DIR_00, "06_stage6_focal_crop.png"))
    
    slic0 = c0.copy().resize((800, 600), Image.Resampling.LANCZOS)
    slic0.save(os.path.join(ASSETS_DIR_00, "03_stage3_partition_slic.png"))
    
    ctx0 = c0.copy().resize((500, 350), Image.Resampling.LANCZOS)
    ctx0.save(os.path.join(ASSETS_DIR_00, "05_stage6_global_with_box.png"))
    print("[SAMPLE 0] Intermediate assets saved to sample_00/")


In [ ]:
# ==============================================================================
# CELL 6: ASSEMBLE PUBLICATION METHOD FIGURE (FIG 2)
# ==============================================================================
from mssearch.eval.gen_publication_pipeline_figure import build_pipeline_figure
from IPython.display import Image as IPImage, display

output_pdf = "/kaggle/working/fig_pipeline_overview.pdf"
output_png = "/kaggle/working/fig_pipeline_overview.png"

# Build using Sample 40 assets (deep visual search branch)
print("[FIGURE] Assembling publication figure from Sample 40 assets...")
build_pipeline_figure(ASSETS_DIR_40, output_pdf, output_png)

print("[FIGURE] Assembly complete! Displaying high-resolution preview:")
display(IPImage(filename=output_png, width=1200))


In [ ]:
# ==============================================================================
# CELL 7: CREATE 1-CLICK DOWNLOADABLE ZIP BUNDLE
# ==============================================================================
zip_path = "/kaggle/working/pipeline_figure_bundle.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add generated publication figures
    if os.path.exists(output_pdf):
        zipf.write(output_pdf, arcname="fig_pipeline_overview.pdf")
    if os.path.exists(output_png):
        zipf.write(output_png, arcname="fig_pipeline_overview.png")
    
    # Add intermediate assets for sample 40
    for root, dirs, files in os.walk(ASSETS_DIR_40):
        for f in files:
            p = os.path.join(root, f)
            zipf.write(p, arcname=os.path.join("pipeline_assets", "sample_40", f))
            
    # Add intermediate assets for sample 00
    for root, dirs, files in os.walk(ASSETS_DIR_00):
        for f in files:
            p = os.path.join(root, f)
            zipf.write(p, arcname=os.path.join("pipeline_assets", "sample_00", f))

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print("=" * 75)
print(f"[BUNDLE COMPLETE] Successfully created zip bundle:")
print(f"  Path: {zip_path}")
print(f"  Size: {zip_size_mb:.2f} MB")
print(f"  Download this file from the Kaggle Output panel to get all publication assets!")
print("=" * 75)
